# 14 - Improved Model | Final XGBoost with feature engineering

## 1. What I am doing
Improve the notebook 12/13 XGBoost using **leakage-safe feature engineering** and early stopping,
then evaluate once on the untouched 2026 test period.

## 2. Why I am doing it
The previous tuned XGBoost (notebook 13) reached 36.5% test accuracy and 0.12 macro F1. Error analysis showed
two unused signals:

* **Reporting-time patterns** - back-dated reports (fraud, theft) are frequently logged at exactly 00:00,
  on the hour, or on the 1st of the month. These come from the `date` input, so no new input is needed.
* **Historical crime mix** - the share of each crime type previously recorded for a location type, beat,
  district, map grid cell and domestic flag. Computed **from training rows only** (out-of-fold inside training),
  so no validation/test labels are used.

The split is identical to notebooks 07-13: train < 2025-07-01, validation Jul-Dec 2025, test >= 2026-01-01.
Leakage columns (`iucr`, `description`, `fbi_code`, `arrest`, `updated_on`, `case_number`) remain excluded.

## 3. Install notebook libraries

%pip install numpy scipy scikit-learn pandas xgboost

In [1]:
import os, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             top_k_accuracy_score, confusion_matrix)
SEED = 42
TRAIN_END, VALIDATION_END, DATA_END = '2025-07-01', '2026-01-01', '2026-09-17'
COLUMNS = ['date', 'primary_type', 'location_description', 'domestic', 'beat', 'district',
           'ward', 'community_area', 'latitude', 'longitude']

## 4. Load the cleaned dataset
Uses `workspace.default.chicago_crime_clean` on Databricks. Outside Databricks it falls back to the same
public source (City of Chicago Data Portal, dataset `ijzp-q8t2`), cached in `CRIME_PORTAL_CACHE`.

In [2]:
def load_data():
    if 'spark' in globals():
        for table in ['workspace.default.chicago_crime_clean', 'default.chicago_crime_clean']:
            try:
                return spark.table(table).select(*COLUMNS).toPandas()
            except Exception:
                pass
    cache = Path(os.environ.get('CRIME_PORTAL_CACHE', 'portal_cache'))
    cache.mkdir(parents=True, exist_ok=True)
    if not list(cache.glob('part_*.csv')):
        base = ('https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$select=id,' + ','.join(COLUMNS) +
                "&$where=date>='2024-01-01T00:00:00'&$order=id&$limit=200000&$offset=")
        for offset in range(0, 800000, 200000):
            pd.read_csv(base + str(offset), dtype=str).to_csv(cache / f'part_{offset}.csv', index=False)
    frame = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(cache.glob('part_*.csv'))])
    return frame.drop_duplicates('id')

df = load_data()
df['date'] = pd.to_datetime(df['date'])
df = df[(df.date >= '2024-01-01') & (df.date < DATA_END)].reset_index(drop=True)
train_m = (df.date < TRAIN_END).values
val_m = ((df.date >= TRAIN_END) & (df.date < VALIDATION_END)).values
test_m = (df.date >= VALIDATION_END).values
print(f'Rows: {len(df):,} | classes: {df.primary_type.nunique()}')
print(f'Train {train_m.sum():,} | validation {val_m.sum():,} | test {test_m.sum():,}')

Rows: 663,675 | classes: 31
Train 376,127 | validation 121,653 | test 165,895


In [3]:
classes = sorted(df.loc[train_m, 'primary_type'].unique())
y = df.primary_type.map({c: i for i, c in enumerate(classes)}).fillna(-1).astype(int).values
K = len(classes)
d = df.date
for c in ['beat', 'district', 'ward', 'community_area']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64').astype(str).replace('<NA>', 'Unknown')
df['location_description'] = df.location_description.fillna('Unknown').str.strip().str.upper()
df['latitude'] = pd.to_numeric(df.latitude, errors='coerce')
df['longitude'] = pd.to_numeric(df.longitude, errors='coerce')
df['domestic_int'] = df.domestic.astype(str).str.lower().isin(['true', '1', '1.0']).astype(int)
med_lat, med_lon = df.loc[train_m, 'latitude'].median(), df.loc[train_m, 'longitude'].median()

def evaluate(y_true, proba):
    pred = proba.argmax(1)
    p, r, f, s = precision_recall_fscore_support(y_true, pred, labels=range(K), zero_division=0)
    keep = y_true >= 0
    return dict(accuracy=accuracy_score(y_true, pred), macro_precision=p.mean(), macro_recall=r.mean(),
                macro_f1=f.mean(), weighted_f1=np.average(f, weights=s),
                top3_accuracy=top_k_accuracy_score(y_true[keep], proba[keep], k=3, labels=range(K))), \
           pd.DataFrame(dict(primary_type=classes, precision=p, recall=r, f1=f, support=s))

## 5. Reference: previous pipeline (notebook 12/13 configuration)
One-hot categorical codes + cyclical time features, XGBoost depth 6, learning rate 0.1, 150 trees.
Re-run here so the previous and final models are compared on exactly the same rows.

In [4]:
X = pd.DataFrame(index=df.index)
X['year'] = d.dt.year
for name, v, per in [('hour', d.dt.hour, 24), ('day_of_week', d.dt.dayofweek, 7), ('month', d.dt.month - 1, 12)]:
    X[name + '_sin'] = np.sin(2 * np.pi * v / per)
    X[name + '_cos'] = np.cos(2 * np.pi * v / per)
X['is_weekend'] = (d.dt.dayofweek >= 5).astype(int)
X['domestic_int'] = df.domestic_int
X['coordinates_missing'] = df.latitude.isna().astype(int)
X['latitude'], X['longitude'] = df.latitude, df.longitude
north, west = df.latitude >= med_lat, df.longitude < med_lon
area = np.select([df.latitude.isna(), north & west, north & ~west, ~north & west],
                 ['Unknown', 'North-West', 'North-East', 'South-West'], 'South-East')
cats = df[['location_description', 'beat', 'district', 'ward', 'community_area']].assign(area_group=area)
enc = OneHotEncoder(handle_unknown='ignore').fit(cats[train_m])
Xs = sparse.hstack([enc.transform(cats), sparse.csr_matrix(X.fillna(0).values)]).tocsr().astype(np.float32)
prev_model = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                       tree_method='hist', n_jobs=-1, random_state=SEED)
prev_model.fit(Xs[train_m], y[train_m])
prev_val, _ = evaluate(y[val_m], prev_model.predict_proba(Xs[val_m]))
prev_test, _ = evaluate(y[test_m], prev_model.predict_proba(Xs[test_m]))
pd.DataFrame({'previous validation': prev_val, 'previous test': prev_test}).round(4)

,previous validation,previous test
accuracy,0.3752,0.3645
macro_precision,0.2574,0.2327
macro_recall,0.1205,0.1205
macro_f1,0.1248,0.1213
weighted_f1,0.3085,0.2965
top3_accuracy,0.6740,0.6530


## 6. Final XGBoost with feature engineering
### 6.1 Time and reporting-pattern features

In [5]:
F = pd.DataFrame(index=df.index)
F['hour'], F['minute'], F['dow'] = d.dt.hour, d.dt.minute, d.dt.dayofweek
F['month'], F['day'] = d.dt.month, d.dt.day
F['is_midnight'] = ((d.dt.hour == 0) & (d.dt.minute == 0)).astype(int)
F['on_the_hour'] = (d.dt.minute == 0).astype(int)
F['first_of_month'] = (d.dt.day == 1).astype(int)
F['is_weekend'] = (d.dt.dayofweek >= 5).astype(int)
F['domestic_int'] = df.domestic_int
F['latitude'], F['longitude'] = df.latitude, df.longitude
F['rot45_a'], F['rot45_b'] = df.latitude + df.longitude, df.latitude - df.longitude
F['coordinates_missing'] = df.latitude.isna().astype(int)
for c in ['beat', 'district', 'ward', 'community_area']:
    F[c + '_num'] = pd.to_numeric(df[c], errors='coerce')
for c in ['location_description', 'beat']:
    F[c + '_freq'] = df[c].map(df.loc[train_m, c].value_counts(normalize=True)).fillna(0)
# Evidence for the reporting-pattern features (training rows only)
train_view = df.loc[train_m].assign(midnight=F.loc[train_m, 'is_midnight'])
display(train_view.groupby('primary_type').midnight.mean().sort_values(ascending=False).head(8).rename('share logged at 00:00'))

primary_type
HUMAN TRAFFICKING             0.606061
OFFENSE INVOLVING CHILDREN    0.198164
CRIMINAL SEXUAL ASSAULT       0.170189
SEX OFFENSE                   0.135002
DECEPTIVE PRACTICE            0.124902
OBSCENITY                     0.112360
INTIMIDATION                  0.099099
OTHER NARCOTIC VIOLATION      0.090909
Name: share logged at 00:00, dtype: float64

### 6.2 Historical crime-mix priors (smoothed target encoding)
For each key, the smoothed share of the 12 most common crime types in **training data**.
Training rows get out-of-fold values (5 folds) so a row never sees its own label;
validation and test rows use full-training statistics only.

In [6]:
Y1 = np.zeros((len(df), K), np.float32)
tr_idx = np.where(train_m)[0]
Y1[tr_idx, y[tr_idx]] = 1
prior = Y1[tr_idx].mean(0)
TOP = np.argsort(-prior)[:12]

def crime_mix(keys, tag, alpha=20):
    keys = pd.Series(keys, index=df.index).astype(str).values
    out = np.zeros((len(df), len(TOP)), np.float32)
    def table(rows):
        g = pd.DataFrame(Y1[rows][:, TOP]).groupby(keys[rows])
        s, n = g.sum(), g.size()
        return pd.DataFrame((s.values + alpha * prior[TOP]) / (n.values[:, None] + alpha), index=s.index)
    def lookup(tbl, rows):
        return tbl.reindex(keys[rows]).fillna(pd.Series(prior[TOP], index=tbl.columns)).values
    other = np.where(~train_m)[0]
    out[other] = lookup(table(tr_idx), other)
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(tr_idx):
        out[tr_idx[b]] = lookup(table(tr_idx[a]), tr_idx[b])
    for j, k in enumerate(TOP):
        F[f'mix_{tag}_{classes[k][:12]}'] = out[:, j]

hour_block = (d.dt.hour // 6).astype(str)
grid = (df.latitude.round(3) // 0.005).astype(str) + '_' + (df.longitude.round(3) // 0.006).astype(str)
crime_mix(df.location_description, 'loc')
crime_mix(df.beat, 'beat')
crime_mix(df.location_description + '|' + df.district, 'locdist')
crime_mix(df.location_description + '|' + hour_block, 'lochr')
crime_mix(df.domestic_int.astype(str) + '|' + df.location_description, 'domloc')
crime_mix(grid, 'grid')
F['location_description'] = df.location_description.astype('category')
F['district'] = df.district.astype('category')
print('Final feature count:', F.shape[1])

Final feature count: 95


## 7. Train the final XGBoost with early stopping on validation
The validation period is used only to choose the number of trees; the test period is not touched.

In [7]:
start = time.perf_counter()
final_model = xgb.XGBClassifier(n_estimators=2000, max_depth=7, learning_rate=0.06, subsample=0.8, colsample_bytree=0.7,
                       min_child_weight=5, reg_lambda=2.0, enable_categorical=True, max_cat_to_onehot=1,
                       tree_method='hist', n_jobs=-1, random_state=SEED, early_stopping_rounds=50,
                       eval_metric='mlogloss')
final_model.fit(F[train_m], y[train_m], eval_set=[(F[val_m], y[val_m])], verbose=200)
print(f'Training time: {time.perf_counter() - start:.0f}s | best iteration: {final_model.best_iteration}')

[0]	validation_0-mlogloss:2.32820


[200]	validation_0-mlogloss:1.83318


[284]	validation_0-mlogloss:1.83337


Training time: 2209s | best iteration: 234


## 8. Final evaluation: previous vs final model

In [8]:
pv, pt = final_model.predict_proba(F[val_m]), final_model.predict_proba(F[test_m])
final_val, _ = evaluate(y[val_m], pv)
final_test, per_class = evaluate(y[test_m], pt)
majority = (y[test_m] == np.bincount(y[train_m]).argmax()).mean()
summary = pd.DataFrame({'previous validation': prev_val, 'final validation': final_val,
                        'previous test': prev_test, 'final test': final_test}).round(4)
print(f'Majority-class (always THEFT) test accuracy: {majority:.4f}')
summary

Majority-class (always THEFT) test accuracy: 0.2159


,previous validation,final validation,previous test,final test
accuracy,0.3752,0.3960,0.3645,0.3844
macro_precision,0.2574,0.2601,0.2327,0.2343
macro_recall,0.1205,0.1491,0.1205,0.1471
macro_f1,0.1248,0.1590,0.1213,0.1530
weighted_f1,0.3085,0.3490,0.2965,0.3365
top3_accuracy,0.6740,0.7015,0.6530,0.6816


In [9]:
display(per_class.sort_values('f1', ascending=False).round(3).head(12))
cm = confusion_matrix(y[test_m], pt.argmax(1), labels=range(K))
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
print(f'Largest error pair: {classes[i]} predicted as {classes[j]} ({cm[i, j]:,} rows)')
importance = pd.Series(final_model.get_booster().get_score(importance_type='gain')).sort_values(ascending=False)
importance.head(15).round(1)

,primary_type,precision,recall,f1,support
2,BATTERY,0.464,0.676,0.550,30522
29,THEFT,0.411,0.650,0.503,35818
23,PROSTITUTION,0.426,0.467,0.445,105
8,DECEPTIVE PRACTICE,0.435,0.400,0.417,9543
10,HOMICIDE,0.924,0.269,0.417,316
16,MOTOR VEHICLE THEFT,0.325,0.518,0.399,13036
17,NARCOTICS,0.432,0.348,0.386,4670
30,WEAPONS VIOLATION,0.181,0.315,0.230,3730
6,CRIMINAL SEXUAL ASSAULT,0.539,0.134,0.215,1287
5,CRIMINAL DAMAGE,0.243,0.148,0.184,18149


Largest error pair: ASSAULT predicted as BATTERY (6,213 rows)


domestic_int               160.0
mix_domloc_MOTOR VEHICL    118.1
mix_domloc_BATTERY         111.0
on_the_hour                 47.7
mix_domloc_DECEPTIVE PR     47.4
mix_locdist_THEFT           46.8
mix_domloc_BURGLARY         37.2
mix_domloc_ASSAULT          30.7
mix_domloc_NARCOTICS        30.4
mix_domloc_THEFT            28.7
mix_domloc_ROBBERY          25.8
mix_locdist_NARCOTICS       25.1
mix_domloc_CRIMINAL DAM     22.9
mix_lochr_THEFT             22.9
mix_domloc_WEAPONS VIOL     22.0
dtype: float64

## 9. Optional: per-class decision weights (macro-F1 trade-off)
Weights are tuned on validation only by coordinate ascent, then applied once to test.
This favours rare classes: macro F1 rises but accuracy falls, so it is **reported, not deployed**.

In [10]:
from sklearn.metrics import f1_score
macro = lambda yy, pp: f1_score(yy, pp, average='macro', labels=range(K), zero_division=0)
w = np.ones(K)
best = macro(y[val_m], pv.argmax(1))
for it in range(3):
    for k in range(K):
        base, chosen = w[k], w[k]
        for g in [0.5, 0.7, 0.85, 1, 1.2, 1.5, 2, 2.5, 3, 4, 6]:
            w[k] = base * g if it else g
            score = macro(y[val_m], (pv * w).argmax(1))
            if score > best + 1e-5:
                best, chosen = score, w[k]
        w[k] = chosen
weighted_test, _ = evaluate(y[test_m], pt * w)
pd.DataFrame({'final test': final_test, 'final test + class weights': weighted_test}).round(4)

,final test,final test + class weights
accuracy,0.3844,0.3533
macro_precision,0.2343,0.1970
macro_recall,0.1471,0.1796
macro_f1,0.1530,0.1764
weighted_f1,0.3365,0.3573
top3_accuracy,0.6816,0.6507


In [11]:
out = Path('artifacts'); out.mkdir(exist_ok=True)
out.joinpath('final_metrics.json').write_text(json.dumps(dict(
    split=dict(train_end=TRAIN_END, validation_end=VALIDATION_END),
    previous=dict(validation=prev_val, test=prev_test), final=dict(validation=final_val, test=final_test),
    final_best_iteration=int(final_model.best_iteration), majority_test_accuracy=majority,
    final_test_with_class_weights=weighted_test), indent=2, default=float))
per_class.to_csv(out / 'final_test_per_class.csv', index=False)

## 10. Findings / conclusion
* The final XGBoost improves **every** metric on validation and on the untouched 2026 test period versus the previous model, on the same rows and split.
* The largest gains come from the domestic x location crime-mix prior and the reporting-time flags.
* Top-3 accuracy matters for the product because the interface returns a ranked list.
* Remaining weakness: rare classes and look-alike pairs (ASSAULT vs BATTERY, CRIMINAL DAMAGE vs THEFT) that the
  available context cannot separate; legally they differ by the act, not by time or place.